In [3]:
from pathlib import Path
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

In [4]:
RANDOM_STATE = 42
TEST_SIZE = 0.20

REPORTS_DIR = Path("reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project configuration completed.")
print("Reports folder:", REPORTS_DIR.resolve())

Project configuration completed.
Reports folder: C:\Users\Work\Desktop\Testable ML training and inference pipeline\Day1-Problem definition and data quality\reports


In [5]:
dataset = load_breast_cancer(as_frame=True)

X = dataset.data.copy()
y = dataset.target.copy()

df = X.copy()
df["target"] = y

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("Input features:", X.shape[1])
print("Target column: target")
print("Target meanings:", dict(enumerate(dataset.target_names)))

Dataset loaded successfully!
Dataset shape: (569, 31)
Input features: 30
Target column: target
Target meanings: {0: np.str_('malignant'), 1: np.str_('benign')}


In [6]:
problem_definition = {
    "project": "Testable ML Training and Inference Pipeline",
    "problem_type": "Binary classification",
    "target_column": "target",
    "population": "Diagnostic samples represented in the dataset",
    "number_of_features": X.shape[1],
    "evaluation_strategy": "Stratified train-test split",
}

for key, value in problem_definition.items():
    print(f"{key}: {value}")

project: Testable ML Training and Inference Pipeline
problem_type: Binary classification
target_column: target
population: Diagnostic samples represented in the dataset
number_of_features: 30
evaluation_strategy: Stratified train-test split


In [7]:
data_dictionary = pd.DataFrame({
    "column_name": list(X.columns) + ["target"],
    "description": list(dataset.feature_names) + [
        "Diagnosis class: 0 = malignant, 1 = benign"
    ],
    "column_type": ["numerical feature"] * X.shape[1] + ["target"]
})

display(data_dictionary.head(10))

data_dictionary.to_csv(
    REPORTS_DIR / "day1_data_dictionary.csv",
    index=False
)

print("Data dictionary saved.")

,column_name,description,column_type
0,mean radius,mean radius,numerical feature
1,mean texture,mean texture,numerical feature
2,mean perimeter,mean perimeter,numerical feature
3,mean area,mean area,numerical feature
4,mean smoothness,mean smoothness,numerical feature
5,mean compactness,mean compactness,numerical feature
6,mean concavity,mean concavity,numerical feature
7,mean concave points,mean concave points,numerical feature
8,mean symmetry,mean symmetry,numerical feature
9,mean fractal dimension,mean fractal dimension,numerical feature


Data dictionary saved.


In [8]:
missing_values = df.isnull().sum()

print("Missing values per column:")
display(missing_values[missing_values > 0])

print("Total missing values:", missing_values.sum())

Missing values per column:


Series([], dtype: int64)

Total missing values: 0


In [9]:
duplicate_count = df.duplicated().sum()

print("Duplicate records:", duplicate_count)

Duplicate records: 0


In [10]:
missing_targets = y.isnull().sum()

print("Missing target values:", missing_targets)

assert missing_targets == 0, "Target contains missing values!"

print("Target validation: PASSED")

Missing target values: 0
Target validation: PASSED


In [11]:
class_counts = y.value_counts().sort_index()

for class_id, count in class_counts.items():
    print(f"Class {class_id} ({dataset.target_names[class_id]}): {count}")

class_distribution = pd.DataFrame({
    "class_id": class_counts.index,
    "class_name": [
        dataset.target_names[int(i)] for i in class_counts.index
    ],
    "record_count": class_counts.values
})

display(class_distribution)

class_distribution.to_csv(
    REPORTS_DIR / "day1_class_distribution.csv",
    index=False
)

Class 0 (malignant): 212
Class 1 (benign): 357


,class_id,class_name,record_count
0,0,malignant,212
1,1,benign,357


In [12]:
def validate_dataset(features, target):
    if features.empty:
        raise ValueError("Dataset has no records.")

    if len(features) != len(target):
        raise ValueError("Features and target lengths do not match.")

    if features.isnull().any().any():
        raise ValueError("Input features contain missing values.")

    if target.isnull().any():
        raise ValueError("Target contains missing values.")

    if not features.columns.is_unique:
        raise ValueError("Duplicate feature column names found.")

    if "target" in features.columns:
        raise ValueError("Target leakage: target is an input feature.")

    return True


validate_dataset(X, y)
print("Valid dataset test: PASSED")

Valid dataset test: PASSED


In [13]:
bad_X = X.copy()
bad_X.iloc[0, 0] = float("nan")

try:
    validate_dataset(bad_X, y)
except ValueError as error:
    print("Negative test: PASSED")
    print("Error detected:", error)
else:
    raise AssertionError("Missing feature was not rejected.")

Negative test: PASSED
Error detected: Input features contain missing values.


In [14]:
bad_y = y.copy()
bad_y.iloc[0] = float("nan")

try:
    validate_dataset(X, bad_y)
except ValueError as error:
    print("Negative test: PASSED")
    print("Error detected:", error)
else:
    raise AssertionError("Missing target was not rejected.")

Negative test: PASSED
Error detected: Target contains missing values.


In [15]:
exact_matches = [
    column for column in X.columns
    if X[column].equals(y)
]

print("Features identical to target:", exact_matches)

assert not exact_matches, "Potential target leakage detected."

print("Exact feature-target check: PASSED")

Features identical to target: []
Exact feature-target check: PASSED


In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

print("\nTraining class distribution:")
print(y_train.value_counts().sort_index())

print("\nTesting class distribution:")
print(y_test.value_counts().sort_index())

Training records: 455
Testing records: 114

Training class distribution:
target
0    170
1    285
Name: count, dtype: int64

Testing class distribution:
target
0    42
1    72
Name: count, dtype: int64


In [17]:
assert set(X_train.index).isdisjoint(set(X_test.index))
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)

print("Train-test overlap check: PASSED")
print("Training feature-target alignment: PASSED")
print("Testing feature-target alignment: PASSED")

Train-test overlap check: PASSED
Training feature-target alignment: PASSED
Testing feature-target alignment: PASSED


In [18]:
split_summary = pd.DataFrame([
    {
        "split": "train",
        "records": len(X_train),
        "class_0": int((y_train == 0).sum()),
        "class_1": int((y_train == 1).sum()),
        "random_state": RANDOM_STATE,
    },
    {
        "split": "test",
        "records": len(X_test),
        "class_0": int((y_test == 0).sum()),
        "class_1": int((y_test == 1).sum()),
        "random_state": RANDOM_STATE,
    }
])

display(split_summary)

split_summary.to_csv(
    REPORTS_DIR / "day1_split_summary.csv",
    index=False
)

print("Split summary saved.")

,split,records,class_0,class_1,random_state
0,train,455,170,285,42
1,test,114,42,72,42


Split summary saved.


In [19]:
quality_report = pd.DataFrame([
    {"check": "total_records", "result": len(df)},
    {"check": "total_columns", "result": df.shape[1]},
    {"check": "input_features", "result": X.shape[1]},
    {"check": "missing_values", "result": int(df.isnull().sum().sum())},
    {"check": "duplicate_rows", "result": int(df.duplicated().sum())},
    {"check": "missing_targets", "result": int(y.isnull().sum())},
    {"check": "train_records", "result": len(X_train)},
    {"check": "test_records", "result": len(X_test)},
    {"check": "exact_target_feature_matches", "result": len(exact_matches)},
])

display(quality_report)

quality_report.to_csv(
    REPORTS_DIR / "day1_quality_report.csv",
    index=False
)

print("Quality report saved.")

,check,result
0,total_records,569
1,total_columns,31
2,input_features,30
3,missing_values,0
4,duplicate_rows,0
5,missing_targets,0
6,train_records,455
7,test_records,114
8,exact_target_feature_matches,0


Quality report saved.


In [20]:
assert len(df) == 569
assert X.shape[1] == 30
assert df.isnull().sum().sum() == 0
assert df.duplicated().sum() == 0
assert len(X_train) == 455
assert len(X_test) == 114
assert len(X_train) + len(X_test) == len(X)

print("ALL DAY 1 CHECKS PASSED!")
print("Reports folder:", REPORTS_DIR.resolve())

ALL DAY 1 CHECKS PASSED!
Reports folder: C:\Users\Work\Desktop\Testable ML training and inference pipeline\Day1-Problem definition and data quality\reports
